### ⚠️ Audit note — cases_7d_avg leakage gap NOT fixable from this notebook

This notebook builds its feature sets via `get_feature_sets()` / `load_canonical_dataset()`, both **imported from `canonical_pipeline.py`**, whose source was not included in the files reviewed. The same `cases_7d_avg` leakage confirmed in Notebooks 08/09 (r≈0.96-0.99 with same-day `dengue_cases`, not matched by any `case_pfx` prefix) likely also exists inside `get_feature_sets()`, since this notebook's own correlation printout flags `cases_7d_avg` as the single highest-correlated feature with same-day cases.

**Action required**: open `canonical_pipeline.py` and confirm `get_feature_sets()` excludes `cases_7d_avg` (and any other raw, non-prefixed case-derived column) from `rf_features` / `dl_feat_features`. This cannot be patched here because the function body is not in this notebook.

# 00 — Setup: Canonical Baselines

**Run this notebook ONCE before running Notebooks 11, 12, 13, or 14.**

Those four notebooks all start with:
```python
with open('processed_data/canonical_data.pkl', 'rb') as f:
    d = pickle.load(f)
```
i.e. they expect the canonical (60-feature DL / 44-feature RF) dataset and the cached SARIMA
fit to already exist on disk. This notebook builds and saves:

- `processed_data/canonical_data.pkl`
- `models/sarima_model.pkl`, `metrics/sarima_results.pkl`

The Random Forest baseline is **not** produced here: the canonical weather-only RF lives in
Notebook 03 (`rf_results.pkl` / `rf_model.pkl`), and Notebook 12 trains its own seed=42 reference
inline. The old `rf_*_fair.pkl` caches were leakage-tainted and were removed on 2026-07-02.

You only need to re-run this if you change the CSV data, change `canonical_pipeline.py`'s
feature engineering, or delete the `processed_data/` / `models/` / `metrics/` folders.


In [1]:
import pickle, time, os, warnings
warnings.filterwarnings('ignore')
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from pmdarima import auto_arima

os.makedirs('processed_data', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('metrics', exist_ok=True)

from canonical_pipeline import load_canonical_dataset

CSV_PATH = 'singapore_dengue_weather_2013_2022.csv'  # adjust path if your CSV lives elsewhere

d = load_canonical_dataset(CSV_PATH)
with open('processed_data/canonical_data.pkl', 'wb') as f:
    pickle.dump(d, f)
print(f"✅ Saved processed_data/canonical_data.pkl")
print(f"   rf_features: {len(d['rf_features'])}, dl_seq_features: {len(d['dl_seq_features'])}, "
      f"dl_feat_features: {len(d['dl_feat_features'])}")
assert len(d['dl_seq_features']) == 60, "Expected 60 sequence features per Thesis Ch4.2.2"
assert len(d['rf_features']) == 44, "Expected 44 fair features per Thesis Ch3.3.1 (was 45 before the 2026-06-21 cases_7d_avg leak fix)"


✅ Saved processed_data/canonical_data.pkl
   rf_features: 44, dl_seq_features: 60, dl_feat_features: 44


## SARIMA (Section 3.3.1 statistical baseline)

In [2]:
print("Training SARIMA...")
t0 = time.time()
sarima_model = auto_arima(d['train_df']['dengue_cases'], seasonal=True, m=7,
                           max_p=2, max_q=2, max_P=1, max_Q=1, max_d=1, max_D=1, D=0,
                           trace=False, error_action='ignore', suppress_warnings=True, stepwise=True)
sarima_pred = sarima_model.predict(n_periods=len(d['test_df']))
s_rmse = np.sqrt(mean_squared_error(d['y_test'], sarima_pred))
s_mae = mean_absolute_error(d['y_test'], sarima_pred)
s_r2 = r2_score(d['y_test'], sarima_pred)
print(f"SARIMA trained in {time.time()-t0:.1f}s: RMSE={s_rmse:.2f}, MAE={s_mae:.2f}, R2={s_r2:.4f}, "
      f"order={sarima_model.order}, seasonal={sarima_model.seasonal_order}")

with open('models/sarima_model.pkl', 'wb') as f:
    pickle.dump(sarima_model, f)
with open('metrics/sarima_results.pkl', 'wb') as f:
    pickle.dump({'model': 'SARIMA', 'RMSE': s_rmse, 'MAE': s_mae, 'R2': s_r2, 'predictions': sarima_pred}, f)
print("✅ Saved models/sarima_model.pkl and metrics/sarima_results.pkl")


Training SARIMA...
SARIMA trained in 12.2s: RMSE=62.52, MAE=37.71, R2=-0.2567, order=(2, 1, 2), seasonal=(1, 0, 1, 7)
✅ Saved models/sarima_model.pkl and metrics/sarima_results.pkl


## Random Forest baseline (provided by Notebook 03 / Notebook 12 — not produced here)

In [3]:
# ---------------------------------------------------------------------------
# Random Forest baseline -- intentionally NOT produced here anymore.
#
# The canonical weather-only RF is trained in Notebook 03 (rf_results.pkl /
# rf_model.pkl, R2 = -0.35, 37 features). The rf_*_fair.pkl caches this cell used
# to write relied on the leakage-tainted 44-feature set (cases_7d_avg included,
# R2 = 0.52) and were removed on 2026-07-02. Notebook 12 now trains its own
# seed=42 RF reference inline, so no cached RF file is needed from this notebook.
# ---------------------------------------------------------------------------
print("RF baseline: see Notebook 03 (canonical) / Notebook 12 (inline reference). "
      "No rf_*_fair.pkl written from this setup notebook.")

RF baseline: see Notebook 03 (canonical) / Notebook 12 (inline reference). No rf_*_fair.pkl written from this setup notebook.


In [4]:
print("="*60)
print("  ✅ ALL CANONICAL BASELINE FILES READY")
print("="*60)
for p in ['processed_data/canonical_data.pkl', 'models/sarima_model.pkl', 'metrics/sarima_results.pkl']:
    exists = os.path.exists(p)
    size_kb = os.path.getsize(p)/1024 if exists else 0
    print(f"  {'✅' if exists else '❌'} {p}  ({size_kb:.1f} KB)")
print("\nRandom Forest is provided by Notebook 03 (canonical) and Notebook 12 (inline reference).")
print("You can now run Notebooks 11, 12, 13, and 14.")

  ✅ ALL CANONICAL BASELINE FILES READY
  ✅ processed_data/canonical_data.pkl  (7705.9 KB)
  ✅ models/sarima_model.pkl  (56050.3 KB)
  ✅ metrics/sarima_results.pkl  (6.3 KB)

Random Forest is provided by Notebook 03 (canonical) and Notebook 12 (inline reference).
You can now run Notebooks 11, 12, 13, and 14.
